# 03 — Final cohort construction

**Input** `outputs/dataset_categorized.csv` from Notebook 02.  
**Outputs** `outputs/dataset_final.csv` and `outputs/dataset_excluded.csv`.

Apply row-level exclusion criteria while **preserving every input column** and recording **all reasons for each excluded participant**. No imputation, corrections, or new clinical classifications are performed here.

**Exclusion criteria**

1. Age below 60 years (the intended older-adult study population).
2. Missing HRV measurement or missing/invalid essential factor (age, sex, SBP, DBP, BMI, or its categorical assignment).
3. Invalid physiological input (e.g., nonpositive RR/HR/BP/BMI, negative HRV variability or spectral power, or SBP not greater than DBP).
4. A major unresolved HR/RR inconsistency: a near-tenfold mismatch or an absolute difference greater than 10 bpm between measured HR and `60000 / mean_RR`. The 10-bpm rule is an **operational QC cutoff**, not a clinical diagnostic threshold. Smaller differences are retained.
5. An exact duplicate of all seven HRV measurements in a different row. **Both/all matching records are provisionally excluded** pending original-record verification; the notebook cannot determine which record is authentic.

Unusual yet plausible physiological values, BMI or BP risk categories, and the `outside_review_range` warnings from Notebook 01 are **not** automatic exclusion criteria.

Each reason is stored as a separate code in `exclusion_reasons` using ` | ` as separator, with `exclusion_count` indicating how many distinct reasons apply.

In [1]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path("../../outputs/dataset_categorized.csv")
OUTPUT_DIR = Path("../../outputs")

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Missing Notebook 02 output: {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE)

hrv_columns = [
    "t2m_pre_mean_rr", "t2m_pre_mean_hr", "t2m_pre_sdnn",
    "t2m_pre_rmssd", "t2m_pre_hf", "t2m_pre_lf", "t2m_pre_vlf",
]
factor_columns = [
    "age", "sex", "bp_systolic", "bp_diastolic", "imc",
    "age_group", "sex_category", "bp_category", "bmi_category", "bmi_reference",
]
required = ["source_excel_row"] + hrv_columns + factor_columns
missing_columns = [col for col in required if col not in df.columns]
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")
if df["source_excel_row"].isna().any() or not df["source_excel_row"].is_unique:
    raise ValueError("source_excel_row must be present and unique")

print(f"Input: {len(df)} participants, {len(df.columns)} columns")

Input: 530 participants, 24 columns


In [2]:
# One boolean column per exclusion criterion. No data values are changed.
flags = pd.DataFrame(index=df.index)

flags["AGE_BELOW_60"] = df["age"] < 60

# A participant may have several missing measurements and therefore several reasons.
for col in hrv_columns + factor_columns:
    flags[f"MISSING_{col.upper()}"] = df[col].isna()

flags["INVALID_SEX_CODE"] = df["sex"].notna() & ~df["sex"].isin([1, 2])
flags["INVALID_AGE"] = df["age"].notna() & ((df["age"] < 0) | (df["age"] > 120))
flags["INVALID_BP"] = (
    (df["bp_systolic"] <= 0) |
    (df["bp_diastolic"] <= 0) |
    (df["bp_systolic"] <= df["bp_diastolic"])
)
flags["INVALID_BMI"] = df["imc"] <= 0
flags["INVALID_MEAN_RR_OR_HR"] = (
    (df["t2m_pre_mean_rr"] <= 0) | (df["t2m_pre_mean_hr"] <= 0)
)
for col in ["t2m_pre_sdnn", "t2m_pre_rmssd", "t2m_pre_hf", "t2m_pre_lf", "t2m_pre_vlf"]:
    flags[f"NEGATIVE_{col.upper()}"] = df[col] < 0

# Exclude major HR/RR inconsistencies, not every small difference.
hr = df["t2m_pre_mean_hr"]
rr = df["t2m_pre_mean_rr"]
valid_hr_rr = (hr > 0) & (rr > 0)
expected_hr = 60000 / rr
ratio = hr * rr / 60000
factor_ten = valid_hr_rr & (
    ratio.between(0.097, 0.103) | ratio.between(9.7, 10.3)
)
major_mismatch = valid_hr_rr & ((hr - expected_hr).abs() > 10)
flags["HR_RR_FACTOR_10"] = factor_ten
flags["HR_RR_MAJOR_MISMATCH"] = major_mismatch & ~factor_ten

# Repeated HRV profiles across distinct source records cannot be resolved here.
complete_hrv = df[hrv_columns].notna().all(axis=1)
flags["IDENTICAL_HRV_PROFILE"] = complete_hrv & df.duplicated(subset=hrv_columns, keep=False)

flags = flags.fillna(False).astype(bool)

print("Exclusion criteria and number of affected rows")
display(flags.sum().loc[lambda x: x > 0].sort_values(ascending=False).rename("n").to_frame())

Exclusion criteria and number of affected rows


,n
IDENTICAL_HRV_PROFILE,16
HR_RR_FACTOR_10,11
MISSING_T2M_PRE_SDNN,11
AGE_BELOW_60,10
MISSING_T2M_PRE_RMSSD,8
HR_RR_MAJOR_MISMATCH,2
MISSING_T2M_PRE_MEAN_HR,1


In [3]:
# Store EVERY applicable reason for each participant.
reason_codes = flags.apply(lambda row: " | ".join(row.index[row]), axis=1)
reason_counts = flags.sum(axis=1)
excluded_mask = reason_counts > 0

# Retained dataset: exactly the same columns as the categorized input.
dataset_final = df.loc[~excluded_mask].copy()

# Excluded dataset: original columns plus all exclusion reasons.
dataset_excluded = df.loc[excluded_mask].copy()
dataset_excluded["exclusion_count"] = reason_counts.loc[excluded_mask].astype(int)
dataset_excluded["exclusion_reasons"] = reason_codes.loc[excluded_mask]

assert len(dataset_final) + len(dataset_excluded) == len(df)
assert dataset_final.columns.tolist() == df.columns.tolist()
assert dataset_final["source_excel_row"].is_unique
assert dataset_excluded["source_excel_row"].is_unique
assert set(dataset_final["source_excel_row"]).isdisjoint(dataset_excluded["source_excel_row"])
assert (dataset_excluded["exclusion_count"] >= 1).all()

print(f"Original participants: {len(df)}")
print(f"Retained participants: {len(dataset_final)}")
print(f"Excluded participants: {len(dataset_excluded)}")
print(f"Excluded with multiple reasons: {(dataset_excluded['exclusion_count'] > 1).sum()}")
print("Examples of excluded records")
display(dataset_excluded[["source_excel_row", "age", "exclusion_count", "exclusion_reasons"]].head(20))

Original participants: 530
Retained participants: 476
Excluded participants: 54
Excluded with multiple reasons: 5
Examples of excluded records


,source_excel_row,age,exclusion_count,exclusion_reasons
35,37,57,2,AGE_BELOW_60 | IDENTICAL_HRV_PROFILE
43,45,59,1,AGE_BELOW_60
48,50,69,1,IDENTICAL_HRV_PROFILE
51,53,67,1,IDENTICAL_HRV_PROFILE
83,85,86,1,IDENTICAL_HRV_PROFILE
90,92,69,1,IDENTICAL_HRV_PROFILE
92,94,72,1,IDENTICAL_HRV_PROFILE
111,113,57,1,AGE_BELOW_60
117,119,55,1,AGE_BELOW_60
118,120,79,1,HR_RR_FACTOR_10


In [4]:
# Exactly two exports. No original records or variables are rewritten.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_FILE = OUTPUT_DIR / "dataset_final.csv"
EXCLUDED_FILE = OUTPUT_DIR / "dataset_excluded.csv"

dataset_final.to_csv(FINAL_FILE, index=False)
dataset_excluded.to_csv(EXCLUDED_FILE, index=False)

print(f"Saved: {FINAL_FILE} ({len(dataset_final)} participants, {dataset_final.shape[1]} columns)")
print(f"Saved: {EXCLUDED_FILE} ({len(dataset_excluded)} participants, {dataset_excluded.shape[1]} columns)")

Saved: ../../outputs/dataset_final.csv (476 participants, 24 columns)
Saved: ../../outputs/dataset_excluded.csv (54 participants, 26 columns)
